In [1]:
%%writefile reduction_operations.cpp
#include <iostream>
#include <vector>
#include <cstdlib>
#include <ctime>
#include <omp.h>
#include <climits>
using namespace std;

int main() {
    int n;
    cout << "Enter number of elements: ";
    cin >> n;

    vector<int> arr(n);

    srand(time(0));
    for (int i = 0; i < n; i++) {
        arr[i] = rand() % 1000;   // random numbers between 0 and 999
    }

    cout << "\nFirst few elements:\n";
    for (int i = 0; i < min(n, 20); i++) {
        cout << arr[i] << " ";
    }
    cout << "\n";

    // ----------------------------------
    // Sequential operations
    // ----------------------------------
    int seq_min = INT_MAX;
    int seq_max = INT_MIN;
    long long seq_sum = 0;
    double seq_avg = 0.0;

    double start = omp_get_wtime();

    for (int i = 0; i < n; i++) {
        if (arr[i] < seq_min) seq_min = arr[i];
        if (arr[i] > seq_max) seq_max = arr[i];
        seq_sum += arr[i];
    }
    seq_avg = (double)seq_sum / n;

    double end = omp_get_wtime();
    double seq_time = end - start;

    // ----------------------------------
    // Parallel operations using reduction
    // ----------------------------------
    int par_min = INT_MAX;
    int par_max = INT_MIN;
    long long par_sum = 0;
    double par_avg = 0.0;

    start = omp_get_wtime();

    #pragma omp parallel for reduction(min:par_min) reduction(max:par_max) reduction(+:par_sum)
    for (int i = 0; i < n; i++) {
        if (arr[i] < par_min) par_min = arr[i];
        if (arr[i] > par_max) par_max = arr[i];
        par_sum += arr[i];
    }

    par_avg = (double)par_sum / n;

    end = omp_get_wtime();
    double par_time = end - start;

    // ----------------------------------
    // Print results
    // ----------------------------------
    cout << "\nSequential Results:\n";
    cout << "Min     = " << seq_min << "\n";
    cout << "Max     = " << seq_max << "\n";
    cout << "Sum     = " << seq_sum << "\n";
    cout << "Average = " << seq_avg << "\n";
    cout << "Time    = " << seq_time << " seconds\n";

    cout << "\nParallel Results:\n";
    cout << "Min     = " << par_min << "\n";
    cout << "Max     = " << par_max << "\n";
    cout << "Sum     = " << par_sum << "\n";
    cout << "Average = " << par_avg << "\n";
    cout << "Time    = " << par_time << " seconds\n";

    cout << "\nSpeedup = " << seq_time / par_time << "\n";

    return 0;
}

Writing reduction_operations.cpp


In [2]:
!g++ -fopenmp reduction_operations.cpp -o reduction_operations
!echo 1000000 | ./reduction_operations

Enter number of elements: 
First few elements:
371 721 489 722 229 962 2 367 391 659 975 404 709 123 887 39 41 799 475 821 

Sequential Results:
Min     = 0
Max     = 999
Sum     = 499510253
Average = 499.51
Time    = 0.00855287 seconds

Parallel Results:
Min     = 0
Max     = 999
Sum     = 499510253
Average = 499.51
Time    = 0.00858253 seconds

Speedup = 0.996544
